# M3 — LLM-assisted annotation with cross-validationFills `hallucination_type` for the corpus and spot-checks the existing binarylabels, using two or three independent LLM judges that must agree.## Read this before running**This does not replace human annotation of the test split.** PRD **D4** requiresthe test split to be 100% human-verified and double-annotated. LLM agreement isnot human verification. Running this notebook does not satisfy D4, and the reportit produces says so.**The binary label is never written by an LLM.** It already exists in the corpus.Judges only *check* it, and disagreements are flagged for you. Letting a modelwrite the label would make the corpus circular: a hallucination detector trainedon an LLM's opinions learns that LLM's blind spots.**What the judges actually decide:** the *error type* on wrong answers(`entity`, `numeric`, `relational`, `contradiction`, `fabricated`, `overclaim`).That is reporting metadata — never trained on, never predicted.

## 1. Where to runEither works:* **Locally** — you already have `requests`; nothing else to install.* **Google Colab / Kaggle** — easiest place to keep API keys.You need **at least two** judges from different providers. Free options:| Provider | Key | Get one at ||---|---|---|| Gemini | `GEMINI_API_KEY` | aistudio.google.com/apikey || Groq | `GROQ_API_KEY` | console.groq.com/keys || OpenRouter | `OPENROUTER_API_KEY` | openrouter.ai/keys (use a `:free` model) |Anthropic and OpenAI work too but have no free tier.

In [ ]:
import os, subprocess, sysfrom pathlib import Path# Point this at the repository root.REPO = Path("/content/Bhibranti") if Path("/content").exists() else Path(".").resolve()os.chdir(REPO)print("working in:", REPO)def run(*args):    p = subprocess.run([sys.executable, "-X", "utf8", *args],                       capture_output=True, text=True, encoding="utf-8")    print(p.stdout or "", p.stderr or "")    return p.returncode

## 2. Set your keysPaste them here, or set them as Colab secrets / environment variables.Leave one blank if you are not using it — you just need two filled in.

In [ ]:
os.environ["GEMINI_API_KEY"] = ""   # freeos.environ["GROQ_API_KEY"]   = ""   # free# os.environ["OPENROUTER_API_KEY"] = ""ready = [k for k in ("GEMINI_API_KEY", "GROQ_API_KEY", "OPENROUTER_API_KEY",                     "ANTHROPIC_API_KEY", "OPENAI_API_KEY") if os.environ.get(k)]print("keys set:", ready)assert len(ready) >= 2, "cross-validation needs at least two providers"

## 3. Rehearse offline firstNo keys, no cost, deterministic mock judges. This proves the pipeline runsbefore you spend any quota.

In [ ]:
run("src/llm_annotate.py", "--dry-run", "--limit", "60")

## 4. The real runRoughly **1,761 items** go to the judges for a type, plus a 400-record sample tospot-check the binary labels. With two judges that is about 4,300 requests.On free tiers this takes roughly **1–3 hours**, mostly waiting on rate limits.Every reply is cached to `data/annotated/llm_round1/_cache.jsonl`, so if thenotebook dies you can rerun this cell and it picks up exactly where it stoppedwithout re-spending anything.`--threshold 0.67` means 2 of 3 judges must agree. With only two judges iteffectively means both must agree.

In [ ]:
run("src/llm_annotate.py",    "--judges", "gemini,groq",     # edit to match your keys    "--task", "both",    "--threshold", "0.67",    "--verify-sample", "400")

## 5. Review what the judges could not settleOpen `data/annotated/llm_round1/flagged_for_human.csv`.Two kinds of row appear there, and they need different treatment:| `why` | What it means | What to put in `your_decision` ||---|---|---|| judges disagree on the error type | they could not agree which category | one of `entity` `numeric` `relational` `contradiction` `fabricated` `overclaim`, or `skip` || judges disagree with the corpus label | they think the existing correct/wrong label is wrong | `agree` if the corpus is right, `dispute` if you think it is genuinely mislabelled |A `dispute` is **not** applied automatically. A wrong binary label is a findingabout the corpus; decide it deliberately and rebuild, rather than patching it.

In [ ]:
import csvrows = list(csv.DictReader(open("data/annotated/llm_round1/flagged_for_human.csv",                                encoding="utf-8-sig")))print(f"{len(rows)} rows need you\n")for r in rows[:5]:    print(f"[{r['item_id']}] {r['why']}  ({r['condition']})")    print("  Q:", r["question"][:110])    print("  A:", r["answer"][:110])    print("  votes:", r["judge_votes"], "\n")

## 6. Merge into the corpus

The human annotation in `data/annotated/round1/` is always the primary source.
`--with-llm` only fills wrong answers that **no human sheet covers** (the 80% of
train outside the spot-check), and marks them `type_source = llm_consensus` so they
can never be mistaken for human work.

`--dry-run` first — it shows exactly what would change and where every type came
from, without touching anything. Backups (`.bak`) are written automatically on
the real run.

In [ ]:
run("src/merge_annotation.py", "--with-llm", "--dry-run")

In [ ]:
# remove --allow-unreviewed once every row in data/annotated/round1/adjudication.csv has a decision
run("src/merge_annotation.py", "--with-llm", "--allow-unreviewed")

## 7. Re-run the gateNever skip this. Changing the corpus without re-auditing is how a shortcut getsin unnoticed. The metadata probe must stay **below 0.60**.

In [ ]:
run("src/audit.py", "--data", "data/splits")

## 8. What this notebook does and does not add

* The test split (D4) is already human double-annotated (κ = 0.865) — this notebook
  does not touch it.
* It can fill `hallucination_type` for train records outside the 20% human
  spot-check. Those rows carry `type_source = llm_consensus`.

When you write this up, say plainly that those types were proposed by LLM judges
under consensus. Describing them as human annotation would not be true, and it is
the kind of thing an examiner checks.